### Symbolic Regression Example: Thermal Correction for UAV Thermal Flights
This notebook applies symbolic regression to derive a thermal correction equation for uncrewed aerial system (UAS) thermal imagery.
Data were generated by Moises Duran for the TREX project and correspond to field campaigns at Utah State University in 2025 and 2026.
The file `example_drone_thermal_camera_correction.csv` includes the following predictor variables and a target variable: 
- UAS tarp temperature
- air temperature
- vapor pressure deficit (VPD)

The target variable is the infrared temperature measurement (`IRT`).


### Running PySR
Importing the `pysr` package will install Julia and the required Julia dependencies on first use.


Import the remaining Python libraries and the `PySRRegressor` class used throughout this notebook.


In [19]:
import pysr
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import skill_metrics as sm
import mpl_scatter_density  # adds projection='scatter_density'

from pysr import PySRRegressor
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import mean_squared_error, r2_score
# from sklearn.model_selection import KFold
from pathlib import Path


### Configure PySR
Define the parameters used to train the symbolic regression model. These settings control search complexity, reproducibility, and optimization behavior.


### Load Dataset
Read the thermal correction dataset and perform any necessary preprocessing before training the symbolic regression model.


In [20]:
df = pd.read_csv('03_df_bands_structure_VI_LAI_NDVIc.csv')

# Define the units for each column (for dimensional analysis)
X_units = ["W/m^2/W/m^2", "W/m^2/W/m^2", "W/m^2/W/m^2", "W/m^2/W/m^2"]  
# TARP, Tair, RH, VPD, Tair_1hr, RH_1hr, VPD_1hr (using "1" for dimensionless)
y_units = "m^2/m^2"  # IRT (target)
# https://ai.damtp.cam.ac.uk/dynamicquantities/stable/units/#Available-units

# Use all columns except the last as predictors and the last column as the target.
X = df.iloc[:, 1:5]
y = df.iloc[:, -2]

X.head()

,Red,Green,Blue,NIR
0,0.035958,0.053953,0.005338,0.342562
1,0.035262,0.053063,0.005320,0.340711
2,0.038239,0.053274,0.005266,0.329618
3,0.039237,0.053812,0.005251,0.324419
4,0.038143,0.049900,0.005562,0.285435


### PySR Parameter Selection
The following variables determine the operators, loss function, and symbolic expression search space used by PySR. Modify them with care to preserve model interpretability and numerical stability.


In [21]:
niterations = 1000  # for short runs, set this to a 100, for longer runs, set this to 10 million
timeout_in_seconds = 3600*1  # this line will make the code stop after 2 hours, change it as desired.
elementwise_loss = "L2DistLoss()"  
#  ("L2DistLoss()" mean square) can be changed to "L1DistLoss()" for mean absolute error, see link at the end for more options
# https://juliaml.github.io/LossFunctions.jl/stable/losses/distance/

binary_operators=["+", "*","-","/","^"] # these are the default binary operators (use these for basic equations)
# activate this line to use simple ratio and normalized diference functions
binary_operators.extend(["ND(x,y) = ((x-y)/(x+y))", 
                         "SR(x,y)=x/y", 
                         "CC(x,y)=x/(x+y)",
                         "ID(x,y)=(1/x)-(1/y)",
                         "NP(x,y)=x*y/(x+y)",
                         "SI(x,y)=(1-x/y)"
                         ]) 
 
unary_operators = ["sqrt","inv","cbrt","cube","square"]  # other basic operators can be added here
unary_operators.extend(["exp", "log"])  # activate this line to use these functions 
# unary_operators.extend(["sin", "cos", "tan"])  # activate this line to use trigonometric functions 

extra_sympy_mappings = {"ND": lambda x, y: ((x- y)/(x+y)),
						"SR": lambda x, y: x/y, 
						"CC": lambda x, y: x/(x+y),
						"ID": lambda x, y: (1/x)-(1/y),
						"NP": lambda x, y: x*y/(x+y),
						"SI": lambda x, y: 1-x/y,
						# "inv": lambda x: 1 / x,
						}  # here the format of the sought expressions are defined

The default PySR parameters below are chosen to improve runtime stability, reproducibility, and model selection consistency.


In [22]:
default_pysr_params = dict(
	model_selection="best", # choose the best tradeoff between accuracy and complexity
	random_state=0, # seed number to ensure reproducible results
	deterministic=True, # preserve deterministic behavior where supported
	parallelism="serial", # serial mode maximizes reproducibility
	maxsize=25, # moderate complexity to reduce search time
	maxdepth=15, # smaller depth limits expression complexity
	# select_k_features=10, # automatically select a smaller predictor subset
	# denoise=True, # use to reduce noise in the response before fitting
	verbosity=0, # suppress verbose output
	parsimony=0.01, # prefer simpler equations
	populations=50, # smaller populations reduce runtime
	weight_randomize=0.2,
	weight_optimize=0.001,
	population_size=50, # smaller candidate set per generation
	ncycles_per_iteration=200, # fewer optimization cycles per iteration
	warm_start=False, # do not reuse previous runs
	elementwise_loss=elementwise_loss, # use the chosen loss function
	procs=1, # one processor for serial execution
	turbo=True, # enable PySR speed optimizations
    # stop early if a simple good equation is found
	early_stop_condition=("stop_if(loss, complexity) = loss < 1e-6 && complexity < 20"), 
    # Amount to penalize dimensional violations:
    dimensional_constraint_penalty=10**5,
    complexity_of_constants=2,
	temp_equation_file=True,  # Put hall of fame in temp dir
    delete_tempfiles=True,   # Delete it afterwards
    adaptive_parsimony_scaling =1000,
)


### Model Training
Instantiate the `PySRRegressor` and fit it to the preprocessed predictor matrix and response vector.


In [ ]:
# Learn equations
model = PySRRegressor(
	niterations=niterations,  
	timeout_in_seconds=timeout_in_seconds,  
	binary_operators=binary_operators, 
	unary_operators=unary_operators, 
	extra_sympy_mappings=extra_sympy_mappings, 
	**default_pysr_params,
)

model.fit(X, y, X_units=X_units, y_units=y_units)

This section visualizes the Pareto front for the discovered equations, illustrating tradeoffs between complexity and loss.


In [ ]:
# Identify the best scoring equation (optimal balance between accuracy and simplicity)
best_score = model.get_best().score
max_index = model.equations_.score == best_score

# Extract Pareto-front data: complexity and loss for all discovered equations
pareto = model.equations_.iloc[:, [0, 1]]

# Plot the Pareto front and highlight the selected best equation
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(pareto.complexity, pareto.loss, 'o-', label='All equations', alpha=0.6)
ax.plot(pareto.complexity[max_index], pareto.loss[max_index], 'ro', markersize=12, label='Best equation')
ax.set_xlabel('Complexity')
ax.set_ylabel('Loss (MSE)')
ax.set_title('Pareto Front: Complexity vs. Loss')
ax.grid(True, alpha=0.3)
ax.legend()

# Annotate each equation index on the plot
for idx, (complexity, loss) in enumerate(zip(pareto.complexity, pareto.loss)):
    ax.annotate(
        f'{idx}',
        xy=(complexity, loss),
        xytext=(0, 10),
        textcoords='offset points',
        ha='center',
        fontsize=9,
    )

plt.tight_layout()
plt.show()


View the equations in SymPy format to examine its symbolic structure and confirm its mathematical form.

In [ ]:
for i in range(len(model.equations_)):
    eq_str = model.sympy(i)
    print(f"Equation {i} (Complexity {model.equations_.loc[i, 'complexity']}):")
    display((eq_str))
    print()  # Add spacing

The following cell displays the symbolic form of any equation in `model.equations_` by selecting its index.


In [ ]:
found_equations = model.equations_.loc[:, ['equation', 'loss']]
# pd.set_option('display.max_colwidth', 200)
# display(found_equations)

# Apply a color ramp to the loss column
styled_df = found_equations.style.background_gradient(
    subset=['loss'],
    cmap='Paired',
)

styled_df

## Results
The plots and tables below summarize the performance of the discovered symbolic regression models.


### Prediction Visualizations
Plot the modeled values versus measured values for the discovered solutions to evaluate prediction quality and identify systematic biases.


### Residual Distribution Analysis
Examine the distribution of prediction residuals (errors) for each discovered equation to assess bias and deviation patterns. Residuals should ideally be centered near zero with minimal spread, indicating unbiased predictions.


In [ ]:
# "Viridis-like" colormap with white background
white_viridis = LinearSegmentedColormap.from_list('white_viridis', [
	(0, '#ffffff'),
	(1e-20, "#0004FF"),
	(0.2, "#0818F6"),
	(0.4, "#009d00"),
	(0.6, "#009d0a"),
	(0.8, "#fc0001"),
	(1, "#ff0000"),
], N=256)

num_eq = len(model.equations_)
fig_rows = int(np.ceil(num_eq / 4))
fig_cols = 4

fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(3*fig_cols, 3*fig_rows),
                         sharey=True, sharex=True,
                         subplot_kw=dict(projection="scatter_density"))

# Compute dynamic axis limits based on measured data range
y_min, y_max = y.min(), y.max()
margin = 0.05 * (y_max - y_min)
axis_limits = (y_min - margin, y_max + margin)

# Collect density objects for shared colorbar
density_objects = []

# Generate predictions and visualize for each equation
for ax, eq_idx in zip(axes.flatten(), range(num_eq)):
	y_pred = model.predict(X, index=eq_idx)
	
	# Create scatter density plot
	density = ax.scatter_density(y_pred, y, cmap=white_viridis)
	density_objects.append(density)
	
	# Plot 1:1 reference line
	ax.plot([axis_limits[0], axis_limits[1]], [axis_limits[0], axis_limits[1]], 
	        linestyle='--', color='k', linewidth=0.5)
	
	# Configure subplot appearance
	ax.set_aspect('equal')
	ax.set_xlim(axis_limits)
	ax.set_ylim(axis_limits)
	ax.grid(True, alpha=0.3)
	ax.set_xlabel("Modeled")
	ax.set_ylabel("Measured")
	
	# Compute and display performance metrics
	mse = mean_squared_error(y_pred, y)
	r2 = r2_score(y, y_pred)
	ax.set_title(f'Eq {eq_idx}\nMSE: {mse:.1f}, R²: {r2:.2f}')

# Adjust figure layout to reserve space for colorbar on the right
fig.subplots_adjust(right=0.88)

# Add colorbar in the reserved space
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
fig.colorbar(density_objects[0], cax=cbar_ax, label='Number of points')

plt.show()


In [ ]:
# Set professional matplotlib style parameters
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.linewidth'] = 0.8
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

fig, axes = plt.subplots(fig_rows, fig_cols, figsize=(3.5*fig_cols, 3.5*fig_rows),
                         sharey=True)

# Generate residual histograms for each equation
for ax, eq_idx in zip(axes.flatten(), range(num_eq)):
	y_pred = model.predict(X, index=eq_idx)
	residuals = y_pred - y
	
	# Make bins symmetric around zero based on data range
	max_abs_residual = max(abs(residuals.min()), abs(residuals.max()))
	bin_interval = max_abs_residual / 10  # Create ~20 bins total (10 on each side)
	bins = np.arange(-max_abs_residual, max_abs_residual + bin_interval, bin_interval)
	
	# Create histogram with professional styling
	n, bins_edges, patches = ax.hist(residuals, bins=bins, density=True, 
	                                   rwidth=0.8, alpha=0.75, color='steelblue', 
	                                   edgecolor='navy', linewidth=1.2)
	
	# Add Gaussian kernel density estimate overlay
	from scipy import stats
	kde = stats.gaussian_kde(residuals)
	x_range = np.linspace(residuals.min(), residuals.max(), 100)
	ax.plot(x_range, kde(x_range), 'r-', linewidth=2, label='KDE', alpha=0.7)
	
	# Configure subplot appearance
	ax.grid(True, alpha=0.4, linestyle='-', linewidth=0.6)
	ax.set_axisbelow(True)
	ax.legend(loc='upper right', fontsize=9)
	
	# Style axes
	for spine in ax.spines.values():
		spine.set_linewidth(1.2)
	
	# Labels with better spacing
	ax.set_xlabel('Prediction Error (var units)', fontweight='bold', fontsize=11)
	ax.set_ylabel('Density', fontweight='bold', fontsize=11)
	
	# Professional title with equation index only
	ax.set_title(f'Equation {eq_idx}', fontweight='bold', fontsize=12, pad=10)
	
	# Compute residual statistics
	mean_residual = np.mean(residuals)
	std_residual = np.std(residuals, ddof=1)
	r2 = r2_score(y, y_pred)
	
	# Add statistics box in upper left corner
	stats_text = f'Mean: {mean_residual:.3f}\nStd: {std_residual:.3f}\nR²: {r2:.3f}\nn = {len(y)}'
	ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
	        fontsize=9, verticalalignment='top', horizontalalignment='left',
	        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85, 
	                  edgecolor='gray', linewidth=1),
	        family='monospace')

# Adjust figure layout
fig.subplots_adjust(right=0.96, hspace=0.35, wspace=0.35)

# Add figure-level title
fig.suptitle('Residual Distribution Analysis: Error Patterns and Normality Assessment', 
             fontsize=14, fontweight='bold', y=0.995)

plt.show()

# Optional: Save at high resolution
# output_dir = Path('outputs')
# output_dir.mkdir(exist_ok=True)
# fig.savefig(output_dir / 'residual_analysis.png', dpi=300, bbox_inches='tight')
# fig.savefig(output_dir / 'residual_analysis.pdf', bbox_inches='tight')


### Taylor Diagram
Construct a Taylor diagram to compare model standard deviation, correlation, and centered root-mean-square difference relative to the reference measurements.


In [ ]:
# number of equations found
n_e = len(model.equations_)

eq_indices = np.arange(n_e, dtype=int)

# Prepare data for Taylor diagram
data = {}

sdev = np.array([])
crmsd = np.array([])
ccoef = np.array([])

data['ref'] = y.values  # Store reference data in data dictionary
taylor_stats = sm.taylor_statistics(data['ref'], data['ref'])
sdev = np.append(sdev, [taylor_stats['sdev'][0]])
crmsd = np.append(crmsd, [taylor_stats['crmsd'][0]])
ccoef = np.append(ccoef, [taylor_stats['ccoef'][0]])

for i in eq_indices:
    ypredict_simpler = model.predict(X, index=i)  # Use integer index
    data[f'pred{i+1}'] = ypredict_simpler
    taylor_stats = sm.taylor_statistics(data[f'pred{i+1}'], data['ref'])
    sdev = np.append(sdev, [taylor_stats['sdev'][1]])
    crmsd = np.append(crmsd, [taylor_stats['crmsd'][1]])
    ccoef = np.append(ccoef, [taylor_stats['ccoef'][1]])

data = pd.DataFrame(data)
label = ['Reference'] + [f'Eq {i}' for i in eq_indices]

intervalsCOR = np.concatenate((np.arange(0, 1.0, 0.2), [0.9, 0.95, 0.99, 1]))
sm.taylor_diagram(
    sdev,
    crmsd,
    ccoef,
    markerLabel=label,
    markerLabelColor='b',
    tickRMS=np.arange(0, 20, 5),
    tickRMSangle=110.0,
    colRMS='m',
    styleRMS=':',
    widthRMS=2.0,
    tickSTD=np.arange(0, 20, 5),
    axismax=20.0,
    colSTD='b',
    styleSTD='-.',
    widthSTD=1.0,
    colCOR='k',
    styleCOR='--',
    widthCOR=1.0,
)

# plt.title('Taylor Diagram of Drone Thermal Camera Correction Models', fontsize=16)


### Bootstrap Evaluation
Estimate uncertainty in equation performance using nonparametric bootstrap resampling.


In [ ]:
# --- Bootstrap evaluation (new cell) ---
# Optional progress bar: if tqdm is not available this line will fail;
# you can remove it or `pip install tqdm` in the environment.
try:
    from tqdm import tqdm
except Exception:
    def tqdm(x):
        return x

# Parameters
B = 100          # number of bootstrap samples (increase for stability)
seed = 0
rng = np.random.default_rng(seed)
n = len(X)
eq_indices = list(model.equations_.index)

# Containers: RMSE and R2 per bootstrap for each equation
scores = {idx: np.empty(B) for idx in eq_indices}
r2_scores = {idx: np.empty(B) for idx in eq_indices}
# If you want prediction intervals, uncomment the next line (may use lots of memory)
# preds = {idx: np.empty((B, n)) for idx in eq_indices}

y_target = y

for b in tqdm(range(B)):
    inds = rng.integers(0, n, size=n)  # sample indices with replacement
    Xb = X.iloc[inds]
    if hasattr(y_target, 'iloc'):
        yb = y_target.iloc[inds]
    else:
        yb = np.asarray(y_target)[inds]
    for idx in eq_indices:
        ypred = model.predict(Xb, index=idx)
        yb_arr = np.asarray(yb).ravel()
        ypred_arr = np.asarray(ypred).ravel()
        scores[idx][b] = np.sqrt(mean_squared_error(yb_arr, ypred_arr))
        r2_scores[idx][b] = r2_score(yb_arr, ypred_arr)
        # preds[idx][b, :] = ypred_arr  # enable to compute per-observation prediction intervals

# Summarize results into a DataFrame
rows = []
for idx in eq_indices:
    rmse_vals = scores[idx]
    r2_vals = r2_scores[idx]
    lo, hi = np.percentile(rmse_vals, [2.5, 97.5])
    r2_lo, r2_hi = np.percentile(r2_vals, [2.5, 97.5])
    rows.append({
        'index': idx,
        'rmse_mean': float(rmse_vals.mean()),
        'rmse_std': float(rmse_vals.std(ddof=1)),
        'rmse_ci_lo': float(lo),
        'rmse_ci_hi': float(hi),
        'r2_mean': float(r2_vals.mean()),
        'r2_std': float(r2_vals.std(ddof=1)),
        'r2_ci_lo': float(r2_lo),
        'r2_ci_hi': float(r2_hi),
        'complexity': model.equations_.loc[idx, 'complexity'],
        'equation': model.equations_.loc[idx, 'equation'],
    })

df_boot = pd.DataFrame(rows)

# Plot bootstrap summaries with subplots
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
indices = df_boot['index'].astype(str).tolist()
x = np.arange(len(df_boot))

rmse_err = np.vstack([
    df_boot['rmse_mean'] - df_boot['rmse_ci_lo'],
    df_boot['rmse_ci_hi'] - df_boot['rmse_mean'],
])
r2_err = np.vstack([
    df_boot['r2_mean'] - df_boot['r2_ci_lo'],
    df_boot['r2_ci_hi'] - df_boot['r2_mean'],
])

axes[0].bar(x, df_boot['rmse_mean'], color='#4c78a8', alpha=0.85, label='RMSE')
axes[0].errorbar(x, df_boot['rmse_mean'], yerr=rmse_err, fmt='none', ecolor='black', capsize=5)
axes[0].set_ylabel('RMSE', fontsize=12, fontweight='bold')
axes[0].set_title('Bootstrap RMSE: mean and 95% confidence intervals', fontsize=14)
axes[0].grid(True, alpha=0.3)

axes[1].bar(x, df_boot['r2_mean'], color='#17becf', alpha=0.85, label='R²')
axes[1].errorbar(x, df_boot['r2_mean'], yerr=r2_err, fmt='none', ecolor='black', capsize=5)
axes[1].set_ylabel('R²', fontsize=12, fontweight='bold')
axes[1].set_title('Bootstrap R²: mean and 95% confidence intervals', fontsize=14)
axes[1].set_xlabel('Equation index', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Eq {i}' for i in indices], rotation=45, ha='right')

fig.tight_layout()

# Display the table and the figure
# display(df_boot)
plt.show()
